In [1]:
import os

from pathlib import Path
os.chdir(Path.cwd().parent)
print(f"Current directory: {Path.cwd()}")

Current directory: /opt/asaldivar/projects/h_lacustris


In [2]:
import importlib.resources
import json
import re

import cobra
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import polars as pl

from tqdm import tqdm

with importlib.resources.open_text("labutils", "plot_template.json") as file:
  style = json.load(file)

#style["layout"]["colorway"] = random.shuffle(style["layout"]["colorway"])
pio.templates["paper"] = go.layout.Template(
    data=style["data"],
    layout=style["layout"],
)
pio.templates.default = "simple_white+paper"

def update_gene_ids(model, gene_map):
    print(f"\nUpdating genes in {model.name}")
    for rxn in tqdm(model.reactions):
        rule = rxn.gene_reaction_rule
        if not rule:
            continue
        gpr = rxn.gpr
        for gene in gpr.genes:
            if not gene in gene_map:
                continue
            old_id = gene
            new_id = gene_map[old_id]
            rule = re.sub(rf"\b{re.escape(old_id)}\b", new_id, rule)
            
        rxn.gene_reaction_rule = rule
        rxn.update_genes_from_gpr()

    print(f"Total genes before: {len(model.genes)}")
    genes_to_remove = [gene.id for gene in model.genes
                               if not bool(gene.reactions)]
    print(f"Removing {len(genes_to_remove)} unused genes")
    for gene_id in genes_to_remove:
        model.genes.remove(gene_id)
    print(f"Total genes after update: {len(model.genes)}")

In [8]:
model_to_red = Path("data/external/") / "model_to_red_dict.csv"
model_to_red_df = pl.read_csv(
    model_to_red,
    has_header = False,
    new_columns = ["model", "red"]
)

In [9]:
model_file = "models/draft/v0.0.1/hlacustris.xml"
model = cobra.io.read_sbml_model(model_file)
model

Name,iHlct
Memory address,7f1596ae6fd0
Number of metabolites,1766
Number of reactions,2294
Number of genes,762
Number of groups,0
Objective expression,1.0*BIOMASS_hlacus_auto - 1.0*BIOMASS_hlacus_auto_reverse_375bd
Compartments,"mitochondrion, chloroplast, cytoplasm, glyoxysome, extracellular, thylakoid"


In [10]:
# The model has some genes with incorret ids
# Used this to fix them
genes_to_fix = {
    'Haem1156': "Haem01156", 'Haem1920': "Haem01920",
    'Haem2499': "Haem02499", 'Haem4107': "Haem04107",
    'Haem5912': "Haem05912", 'Haem8125': "Haem08125"
}
update_gene_ids(model, genes_to_fix)
model_to_red_map = dict(zip(model_to_red_df["model"], model_to_red_df["red"]))
update_gene_ids(model, model_to_red_map)


Updating genes in H. lacustris v0.0.1


100%|█████████████████████████████████████████████████████████| 2294/2294 [00:00<00:00, 17630.52it/s]


Total genes before: 766
Removing 6 unused genes
Total genes after update: 760

Updating genes in H. lacustris v0.0.1


100%|█████████████████████████████████████████████████████████| 2294/2294 [00:00<00:00, 14260.19it/s]


Total genes before: 1519
Removing 759 unused genes
Total genes after update: 760


In [12]:
out_dir = Path("models/draft/v0.0.2")
out_dir.mkdir(parents=True, exist_ok=True)
cobra.io.write_sbml_model(model, out_dir / "hlacustris.xml")